# Baseline model using GCP's AutoML feature

### Libraries

In [1]:
from google.cloud import aiplatform
from google.cloud import storage

import pandas as pd
from datetime import datetime
import re

In [45]:
DATANAME = "synthetic_fashion_demand"
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")

### Initialize Vertex AI

In [2]:
project = !gcloud config get-value project
PROJECT_ID = project[0]
print(PROJECT_ID)
BUCKET = PROJECT_ID
REGION = 'us-central1'

aiplatform.init(project=PROJECT_ID, location=REGION)

fashionmvforecast


### Read CSV and convert into Vertex AI TabularDataset

In [49]:
# Reference or create your managed Vertex AI TabularDataset from GCS

def get_or_create_timeseries_dataset(display_name: str, gcs_path: str) -> aiplatform.TimeSeriesDataset:
    """
    Checks if a TimeSeriesDataset with display_name exists. 
    If found, returns the existing dataset; otherwise, creates a new one.
    """
    # 1. Search for existing datasets matching the display name
    existing_datasets = aiplatform.TimeSeriesDataset.list(
        filter=f'display_name="{display_name}"'
    )
    
    if existing_datasets:
        print(f"Dataset '{display_name}' already exists. Retrieving existing resource...")
        dataset = existing_datasets[0]
        print(f"Resource Name: {dataset.resource_name}")
        return dataset
    
    # 2. If it does not exist, read columns for logging and create it
    print(f"Dataset '{display_name}' not found. Creating new TimeSeriesDataset...")
    
    # Read headers to confirm path accessibility
    columns = pd.read_csv(gcs_path, nrows=0).columns.tolist()
    print(f"Loaded {len(columns)} columns from GCS: {columns[:5]}...")

    dataset = aiplatform.TimeSeriesDataset.create(
        display_name=display_name,
        gcs_source=[gcs_path],
    )
    print(f"Successfully created dataset: {dataset.resource_name}")
    return dataset

# --- Usage ---
gcs_path = f"gs://{BUCKET}/data/input/synthetic_fashion_demand_features.csv"
dataset_name = "synthetic_fashion_demand"

dataset = get_or_create_timeseries_dataset(
    display_name=dataset_name,
    gcs_path=gcs_path
)

Dataset 'synthetic_fashion_demand' already exists. Retrieving existing resource...
Resource Name: projects/456832640267/locations/us-central1/datasets/5214627683652075520


In [ ]:
from functions.Features_Functions import (
    TARGET_COLUMN, TIME_COLUMN, SERIES_ID_COLUMN,
    AVAILABLE_AT_FORECAST, UNAVAILABLE_AT_FORECAST,
    FORECAST_HORIZON, build_column_specs,
)

COLUMN_SPECS = build_column_specs()


### Instantiate the AutoML Forecasting Training Job

In [51]:
forecasting_job = aiplatform.AutoMLForecastingTrainingJob(
    display_name=f"{PROJECT_ID}_baseline_model_AutoML_{TIMESTAMP}",
    optimization_objective="minimize-mae",  # Options: "minimize-rmse", "minimize-mae", "minimize-wape"
    column_specs=COLUMN_SPECS,
)

### Execute the Training Job

In [ ]:
model = forecasting_job.run(
    dataset=dataset,
    target_column=TARGET_COLUMN,
    time_column=TIME_COLUMN,
    time_series_identifier_column=SERIES_ID_COLUMN,
    available_at_forecast_columns=AVAILABLE_AT_FORECAST,
    unavailable_at_forecast_columns=UNAVAILABLE_AT_FORECAST,
    forecast_horizon=FORECAST_HORIZON,  # from params.yaml via functions.Features_Functions
    data_granularity_unit="week",
    data_granularity_count=1,
    budget_milli_node_hours=1000,  # 1 node hour budget
    model_display_name=f"automl_baseline_model_{PROJECT_ID}_{TIMESTAMP}",
    predefined_split_column_name="splits",  # Uses your train/val/test column if provided
)

View Training:
https://console.cloud.google.com/agent-platform/locations/us-central1/training/8873776413518331904?project=456832640267
AutoMLForecastingTrainingJob projects/456832640267/locations/us-central1/trainingPipelines/8873776413518331904 current state:
PIPELINE_STATE_RUNNING
AutoMLForecastingTrainingJob projects/456832640267/locations/us-central1/trainingPipelines/8873776413518331904 current state:
PIPELINE_STATE_RUNNING
AutoMLForecastingTrainingJob projects/456832640267/locations/us-central1/trainingPipelines/8873776413518331904 current state:
PIPELINE_STATE_RUNNING
AutoMLForecastingTrainingJob projects/456832640267/locations/us-central1/trainingPipelines/8873776413518331904 current state:
PIPELINE_STATE_RUNNING
AutoMLForecastingTrainingJob projects/456832640267/locations/us-central1/trainingPipelines/8873776413518331904 current state:
PIPELINE_STATE_RUNNING
AutoMLForecastingTrainingJob projects/456832640267/locations/us-central1/trainingPipelines/8873776413518331904 current s

# Get metrics from model

### check if model exists, and reinatilize 

In [7]:
PROJECT_ID = PROJECT_ID
LOCATION = REGION

TRAINING_PIPELINE_NAME = (
    "projects/" + PROJECT_ID + "/"
    "locations/" + LOCATION + "/"
    "trainingPipelines/8873776413518331904"
)

aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION
)

forecasting_job = aiplatform.AutoMLForecastingTrainingJob.get(
    resource_name=TRAINING_PIPELINE_NAME
)

print(f"Training pipeline: {forecasting_job.resource_name}")
print(f"State: {forecasting_job.state}")

Training pipeline: projects/456832640267/locations/us-central1/trainingPipelines/8873776413518331904
State: 4


In [8]:
model = forecasting_job.get_model()
print(f"Model Resource Name: {model.resource_name}")

AutoMLForecastingTrainingJob run completed. Resource name: projects/456832640267/locations/us-central1/trainingPipelines/8873776413518331904
Model available at projects/456832640267/locations/us-central1/models/2748753230916747264
Model Resource Name: projects/456832640267/locations/us-central1/models/2748753230916747264


### Extract Evaluation Metrics (RMSE, MAE, MAPE, WAPE)

In [9]:
# Fetch evaluation metrics from Vertex AI
evaluations = model.list_model_evaluations()

for evaluation in evaluations:
    eval_dict = evaluation.to_dict()
    metrics = eval_dict.get("metrics", {})
    
    print("=" * 40)
    print(f"Evaluation Display Name: {eval_dict.get('displayName')}")
    print("=" * 40)
    
    # Extract standard regression / forecasting metrics
    for metric_name in ["rootMeanSquaredError", "meanAbsoluteError", "meanAbsolutePercentageError", "rSquared"]:
        if metric_name in metrics:
            print(f"{metric_name:30s}: {metrics[metric_name]:.4f}")

Evaluation Display Name: None
rootMeanSquaredError          : 16.5404
meanAbsoluteError             : 8.7645
meanAbsolutePercentageError   : 48.9652
rSquared                      : 0.5741


### feature importances (attributions) using sampled Shapley values

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Fetch model evaluations
evaluations = model.list_model_evaluations()

# 2. Extract feature attributions (Shapley values) from the evaluation payload
feature_attributions = None
for evaluation in evaluations:
    eval_dict = evaluation.to_dict()
    # Feature attributions are stored inside the modelExplanation key
    if "modelExplanation" in eval_dict:
        attributions = eval_dict["modelExplanation"].get("meanAttributions", [])
        if attributions:
            # Extract feature values
            feature_attributions = attributions[0].get("featureAttributions", {})
            break

# If not in evaluation, attempt extraction directly from the main model object
if not feature_attributions and hasattr(model, "gca_resource"):
    try:
        explanation = model.gca_resource.model_explanation
        feature_attributions = explanation.mean_attributions[0].feature_attributions
    except Exception:
        pass

# 3. Format into a Pandas DataFrame
if feature_attributions:
    importance_df = pd.DataFrame(
        list(feature_attributions.items()), 
        columns=["Feature", "Importance"]
    ).sort_values(by="Importance", ascending=False)

    # 4. Plot top 20 features
    plt.figure(figsize=(10, 8))
    sns.set_theme(style="whitegrid")
    
    # Render horizontal bar plot for top features
    ax = sns.barplot(
        data=importance_df.head(20),
        x="Importance",
        y="Feature",
        palette="viridis"
    )
    
    plt.title("Top 20 Drivers of Demand Forecast (Vertex AI Feature Attributions)", fontsize=13, fontweight="bold")
    plt.xlabel("Mean Absolute Feature Attribution (Shapley Value)", fontsize=11)
    plt.ylabel("Feature Name", fontsize=11)
    
    # Add numerical labels to each bar
    for p in ax.patches:
        width = p.get_width()
        ax.annotate(
            f"{width:.4f}",
            (width, p.get_y() + p.get_height() / 2.),
            ha="left", va="center",
            xytext=(5, 0), textcoords="offset points",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()
else:
    print("No feature attributions were returned. Verify model explanations were enabled during job execution.")